# Purview Copilot — Silver + Gold logic (self-contained)

All transformation code is in this notebook (no `Common/transforms.py`).

**DATA_SOURCE** (settings cell):
- `"demo"` — 3-row in-memory sample (for presentations)
- `"local_parquet"` — bronze from local CSV→parquet path (same as `NB_Purview_Copilot_BronzeToSilver`)
- `"lakehouse"` — `spark.table(Bronze...)` (only if bronze lives in the lakehouse)

In [ ]:
from pathlib import Path

# demo | local_parquet | lakehouse
DATA_SOURCE = "demo"

# --- local_parquet: output of NB_Purview_Copilot_BronzeToSilver (pandas CSV loader) ---
BRONZE_PARQUET_DIR = Path.home() / ".fabric" / "merge_staging" / "_break" / "purview_copilot_bronze"

# --- lakehouse tables (Okta + HC; bronze only if DATA_SOURCE == lakehouse) ---
BRONZE_PURVIEW = "Bronze.purview_copilot_usage"
SILVER_OKTA = "Silver.okta_user_for_ai"
SILVER_HC = "Silver.hc_silver_historical"

MIN_CREATION_DATE = "2026-09-01"  # prod: 2026-10-01
CREATION_MONTH = "2026-09"  # None = no extra month filter

COPILOT_ASSET_ID = "PENDING"
COPILOT_ASSET_NAME = "M365 Copilot"
COMPARE_OLD_INNER_JOIN = True  # False in demo to run faster
SHOW_ROWS = 20

In [ ]:
from pyspark.sql import DataFrame, functions as F

try:
    spark
except NameError:
    from pyspark.sql import SparkSession

    spark = SparkSession.builder.appName("PurviewCopilotLogic").getOrCreate()

if DATA_SOURCE == "demo":
    spark.conf.set("spark.sql.shuffle.partitions", "2")
    spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")
    COMPARE_OLD_INNER_JOIN = False

In [ ]:
OKTA_COLS = [
    "OktaUserId", "OktaUserEmail", "OktaUserFullEmail",
    "OktaUserEmployeeNumber", "OktaUserFullName", "OktaUserRegion", "OktaUserCountry",
]


def _bronze_col(df: DataFrame, *names: str):
    lower = {c.lower(): c for c in df.columns}
    for n in names:
        if n.lower() in lower:
            return F.col(lower[n.lower()])
    raise ValueError(f"Missing column (any of): {names}")


def email_fallback(email: F.Column) -> F.Column:
    return F.when(email.rlike(r"^10[^@]*@"), F.substring(email, 3, 512))


def prompt_count_from_audit_json(messages_json: F.Column) -> F.Column:
    """Count \"isPrompt\":true in Messages JSON (string ops only — no lambda filter / broken regexp_count)."""
    blob = F.coalesce(messages_json, F.lit(""))
    sentinel = "\u0001PROMPT\u0001"
    marked = F.regexp_replace(blob, r'"isPrompt"\s*:\s*true', sentinel)
    return F.greatest(F.size(F.split(marked, sentinel)) - F.lit(1), F.lit(0)).cast("int")


def join_okta_inner_old(base: DataFrame, okta: DataFrame, email_col: str) -> DataFrame:
    u = F.col(email_col)
    a = base.join(okta, u == okta.OktaUserEmail, "inner")
    b = base.join(okta, u == okta.OktaUserFullEmail, "inner")
    return a.unionByName(b).dropDuplicates(["purview_copilot_usage_record_id"])


def join_okta_left_with_fallback(
    base: DataFrame, okta: DataFrame, email_col: str, fallback_col: str
) -> DataFrame:
    user, fb = F.col(email_col), F.col(fallback_col)

    def _coalesce_prefixed(df: DataFrame, prefixes: list[str]) -> DataFrame:
        out = df
        for name in OKTA_COLS:
            out = out.withColumn(name, F.coalesce(*[F.col(f"{p}{name}") for p in prefixes]))
        drop = [f"{p}{c}" for p in prefixes for c in OKTA_COLS]
        return out.drop(*[c for c in drop if c in out.columns])

    p1 = okta.select(*[F.col(c).alias(f"_p1_{c}") for c in OKTA_COLS])
    p2 = okta.select(*[F.col(c).alias(f"_p2_{c}") for c in OKTA_COLS])
    step = base.join(p1, user == F.col("_p1_OktaUserEmail"), "left")
    step = step.join(p2, user == F.col("_p2_OktaUserFullEmail"), "left")
    step = _coalesce_prefixed(step, ["_p1_", "_p2_"])
    f1 = okta.select(*[F.col(c).alias(f"_f1_{c}") for c in OKTA_COLS])
    f2 = okta.select(*[F.col(c).alias(f"_f2_{c}") for c in OKTA_COLS])
    step = step.join(f1, fb == F.col("_f1_OktaUserEmail"), "left")
    step = step.join(f2, fb == F.col("_f2_OktaUserFullEmail"), "left")
    for name in OKTA_COLS:
        step = step.withColumn(
            name,
            F.when(F.col("OktaUserId").isNotNull(), F.col(name)).otherwise(
                F.coalesce(F.col(f"_f1_{name}"), F.col(f"_f2_{name}"))
            ),
        )
    drop_fb = [f"_f1_{c}" for c in OKTA_COLS] + [f"_f2_{c}" for c in OKTA_COLS]
    return step.drop(*[c for c in drop_fb if c in step.columns])


def interaction_type(app_identity: F.Column, agent_name: F.Column) -> F.Column:
    return (
        F.when(app_identity.like("Copilot.Fabric.%"), F.lit("Power BI / Fabric"))
        .when(app_identity.like("Copilot.Studio.%"), F.lit("Copilot Studio Agent"))
        .when(agent_name.isin("Analyst", "Researcher"), F.lit("Specialized Agent"))
        .when(app_identity.like("MicrosoftAgent.%"), F.lit("Built-in Agent"))
        .when(app_identity.isin("Copilot.M365Copilot.WebChat", "Copilot.M365Copilot.Bizchat"), F.lit("Web Chat"))
        .when(app_identity == "Copilot.M365Copilot.Apps", F.lit("Microsoft 365 App"))
        .otherwise(F.lit("Other"))
    )


def _hc_null_columns(joined: DataFrame) -> DataFrame:
    return (
        joined.withColumn("hc_business_unit", F.lit(None).cast("string"))
        .withColumn("hc_job_title", F.lit(None).cast("string"))
        .withColumn("hc_division", F.lit(None).cast("string"))
    )


def attach_hc(joined: DataFrame, hc: DataFrame | None) -> DataFrame:
    # Demo: pass hc=None (never call rdd.isEmpty/count — crashes Windows PySpark workers)
    if hc is None:
        return _hc_null_columns(joined)
    hc_s = hc.select(
        F.col("HCKeyEmail"),
        F.col("HCBusinessUnit").alias("hc_business_unit"),
        F.col("HCJobTitle").alias("hc_job_title"),
        F.col("HCDivision").alias("hc_division"),
    )
    hc_alt = hc_s.select(
        F.col("HCKeyEmail").alias("_hc_alt_key"),
        F.col("hc_business_unit").alias("_hc_alt_bu"),
        F.col("hc_job_title").alias("_hc_alt_job"),
        F.col("hc_division").alias("_hc_alt_div"),
    )
    out = joined.join(hc_s, F.col("purview_copilot_usage_key_email") == hc_s.HCKeyEmail, "left")
    out = out.join(hc_alt, F.col("purview_copilot_usage_key_full_email") == hc_alt._hc_alt_key, "left")
    return (
        out.withColumn("hc_business_unit", F.coalesce("hc_business_unit", "_hc_alt_bu"))
        .withColumn("hc_job_title", F.coalesce("hc_job_title", "_hc_alt_job"))
        .withColumn("hc_division", F.coalesce("hc_division", "_hc_alt_div"))
        .drop("HCKeyEmail", "_hc_alt_key", "_hc_alt_bu", "_hc_alt_job", "_hc_alt_div")
    )


def build_silver(
    bronze: DataFrame,
    okta: DataFrame,
    hc: DataFrame | None,
    min_creation_date: str,
    *,
    use_old_okta_inner_join: bool = False,
) -> DataFrame:
    audit = _bronze_col(bronze, "AuditData", "audit_data").cast("string")
    user_id = _bronze_col(bronze, "UserId", "user_id")
    creation_date = F.to_timestamp(F.trim(_bronze_col(bronze, "CreationDate", "creation_date"))).cast("date")
    messages_json = F.get_json_object(audit, "$.CopilotEventData.Messages")
    app_identity = F.get_json_object(audit, "$.AppIdentity")
    agent_name = F.get_json_object(audit, "$.AgentName")

    base = (
        bronze.filter(creation_date >= F.lit(min_creation_date))
        .select(
            _bronze_col(bronze, "RecordId", "record_id").alias("purview_copilot_usage_record_id"),
            creation_date.alias("purview_copilot_usage_date"),
            F.lower(F.trim(user_id)).alias("purview_copilot_usage_email"),
            app_identity.alias("purview_copilot_usage_app_identity"),
            agent_name.alias("purview_copilot_usage_agent_name"),
            messages_json.alias("_messages_json"),
        )
        .withColumn("purview_copilot_usage_prompt_count", prompt_count_from_audit_json(F.col("_messages_json")))
        .drop("_messages_json")
        .dropDuplicates(["purview_copilot_usage_record_id"])
    )
    base = base.withColumn("purview_copilot_usage_email_fallback", email_fallback(F.col("purview_copilot_usage_email")))
    base = base.withColumn(
        "purview_copilot_usage_interaction_type",
        interaction_type(F.col("purview_copilot_usage_app_identity"), F.col("purview_copilot_usage_agent_name")),
    ).withColumn(
        "purview_copilot_usage_month_no",
        (F.year("purview_copilot_usage_date") * 100 + F.month("purview_copilot_usage_date")).cast("int"),
    )
    okta_s = okta.select(
        F.col("OktaUserId"),
        F.lower(F.trim(F.col("OktaUserEmail"))).alias("OktaUserEmail"),
        F.lower(F.trim(F.col("OktaUserFullEmail"))).alias("OktaUserFullEmail"),
        F.col("OktaUserEmployeeNumber"), F.col("OktaUserFullName"),
        F.col("OktaUserRegion"), F.col("OktaUserCountry"),
    )
    if use_old_okta_inner_join:
        joined = join_okta_inner_old(base, okta_s, "purview_copilot_usage_email")
    else:
        joined = join_okta_left_with_fallback(base, okta_s, "purview_copilot_usage_email", "purview_copilot_usage_email_fallback")
        joined = joined.dropDuplicates(["purview_copilot_usage_record_id"])
    joined = joined.withColumn(
        "purview_copilot_usage_key_email",
        F.when(F.col("OktaUserEmail").isNotNull(), F.concat(F.col("purview_copilot_usage_month_no").cast("string"), F.lit("/"), F.col("OktaUserEmail"))),
    ).withColumn(
        "purview_copilot_usage_key_full_email",
        F.when(F.col("OktaUserFullEmail").isNotNull(), F.concat(F.col("purview_copilot_usage_month_no").cast("string"), F.lit("/"), F.col("OktaUserFullEmail"))),
    )
    joined = attach_hc(joined, hc)
    return joined.select(
        "purview_copilot_usage_record_id", "purview_copilot_usage_date",
        "purview_copilot_usage_email", "purview_copilot_usage_email_fallback",
        "purview_copilot_usage_prompt_count", "purview_copilot_usage_interaction_type",
        "purview_copilot_usage_month_no",
        F.col("OktaUserId").alias("okta_user_id"),
        F.col("OktaUserEmail").alias("okta_user_email"),
        F.col("OktaUserFullEmail").alias("okta_user_full_email"),
        F.col("OktaUserFullName").alias("okta_user_full_name"),
        "hc_business_unit", "hc_job_title", "hc_division",
    )


def build_gold_fact(silver: DataFrame) -> DataFrame:
    grouped = (
        silver.groupBy("purview_copilot_usage_date", "okta_user_id", "purview_copilot_usage_interaction_type")
        .agg(
            F.first("purview_copilot_usage_month_no", ignorenulls=True).cast("int").alias("Month_No"),
            F.sum("purview_copilot_usage_prompt_count").cast("int").alias("Request"),
            F.first("okta_user_email", ignorenulls=True).alias("userEmail"),
            F.first("okta_user_full_email", ignorenulls=True).alias("userFullEmail"),
            F.first("okta_user_full_name", ignorenulls=True).alias("username"),
            F.first("hc_business_unit", ignorenulls=True).alias("User_UN"),
            F.first("hc_job_title", ignorenulls=True).alias("JobRoles"),
        )
        .filter(F.col("Request") >= 1)
    )
    use_case = (
        F.when(F.col("purview_copilot_usage_interaction_type") == "Web Chat", "UC-9007-COP-001")
        .when(F.col("purview_copilot_usage_interaction_type") == "Microsoft 365 App", "UC-9007-COP-002")
        .when(F.col("purview_copilot_usage_interaction_type") == "Built-in Agent", "UC-9007-COP-003")
        .when(F.col("purview_copilot_usage_interaction_type") == "Copilot Studio Agent", "UC-9007-COP-004")
        .when(F.col("purview_copilot_usage_interaction_type") == "Specialized Agent", "UC-9007-COP-005")
        .when(F.col("purview_copilot_usage_interaction_type") == "Power BI / Fabric", "UC-9007-COP-006")
        .otherwise("UC-9007-COP-007")
    )
    return grouped.select(
        "Month_No", "Request", F.lit(COPILOT_ASSET_ID).alias("assetId"), F.lit(COPILOT_ASSET_NAME).alias("assetName"),
        F.col("okta_user_id").alias("userId"), "username", "userEmail", "userFullEmail",
        use_case.alias("useCaseCode"), F.col("purview_copilot_usage_interaction_type").alias("useCaseDisplayName"),
        F.lit(1).alias("UserWithActivity"), F.col("purview_copilot_usage_date").alias("LastModification"),
        "User_UN", "JobRoles",
    )


def display_df(df: DataFrame, n: int = SHOW_ROWS) -> None:
    """Driver-side preview (avoids fragile .show() on some Windows PySpark setups)."""
    pdf = df.limit(n).toPandas()
    try:
        from IPython.display import display

        display(pdf)
    except Exception:
        print(pdf.to_string())


def distinct_count(df: DataFrame, col: str) -> int:
    return df.agg(F.countDistinct(col)).first()[0]

In [ ]:
if DATA_SOURCE == "demo":
    bronze = spark.createDataFrame(
        [
            ("r1", "2026-09-15T10:00:00", "10jane.doe@corp.com",
             '{"AppIdentity":"Copilot.M365Copilot.WebChat","CopilotEventData":{"Messages":[{"isPrompt":true}]}}'),
            ("r2", "2026-09-16T10:00:00", "bob.smith@corp.com",
             '{"AppIdentity":"Copilot.M365Copilot.Apps","CopilotEventData":{"Messages":[{"isPrompt":true}]}}'),
            ("r3", "2026-09-17T10:00:00", "10no.okta@corp.com",
             '{"AppIdentity":"Copilot.M365Copilot.WebChat","CopilotEventData":{"Messages":[{"isPrompt":true}]}}'),
        ],
        ["RecordId", "CreationDate", "UserId", "AuditData"],
    ).coalesce(1)
    okta = spark.createDataFrame(
        [
            ("okta-jane", "jane.doe@corp.com", "jane.doe@corp.com", "E1", "Jane Doe", "EMEA", "MA"),
            ("okta-bob", "bob.smith@corp.com", "bob.smith@corp.com", "E2", "Bob Smith", "EMEA", "MA"),
        ],
        ["OktaUserId", "OktaUserEmail", "OktaUserFullEmail", "OktaUserEmployeeNumber",
         "OktaUserFullName", "OktaUserRegion", "OktaUserCountry"],
    ).coalesce(1)
    hc = None  # skip HC join in demo (empty DataFrame + isEmpty() breaks local Windows Spark)
    print("DATA_SOURCE=demo (3 bronze rows, HC skipped)")

elif DATA_SOURCE == "local_parquet":
    if not BRONZE_PARQUET_DIR.is_dir():
        raise FileNotFoundError(
            f"Bronze parquet not found: {BRONZE_PARQUET_DIR}\n"
            "Run NB_Purview_Copilot_BronzeToSilver with PURVIEW_BRONZE_SOURCE=local_csv first."
        )
    bronze = spark.read.parquet(str(BRONZE_PARQUET_DIR))
    okta = spark.table(SILVER_OKTA)
    hc = spark.table(SILVER_HC)
    print(f"DATA_SOURCE=local_parquet from {BRONZE_PARQUET_DIR}")

elif DATA_SOURCE == "lakehouse":
    bronze = spark.table(BRONZE_PURVIEW)
    okta = spark.table(SILVER_OKTA)
    hc = spark.table(SILVER_HC)
    print(f"DATA_SOURCE=lakehouse bronze={BRONZE_PURVIEW}")
else:
    raise ValueError("DATA_SOURCE must be demo | local_parquet | lakehouse")

if CREATION_MONTH:
    c = F.to_timestamp(F.trim(_bronze_col(bronze, "CreationDate", "creation_date"))).cast("date")
    start, end = F.lit(f"{CREATION_MONTH}-01"), F.add_months(F.lit(f"{CREATION_MONTH}-01"), 1)
    bronze = bronze.filter((c >= start) & (c < end))

In [ ]:
silver_new = build_silver(bronze, okta, hc, MIN_CREATION_DATE, use_old_okta_inner_join=False)

print("Silver (NEW left + fallback):")
display_df(
    silver_new.select(
        "purview_copilot_usage_email", "purview_copilot_usage_email_fallback",
        "okta_user_id", "okta_user_email", "purview_copilot_usage_prompt_count",
    )
)

if COMPARE_OLD_INNER_JOIN:
    silver_old = build_silver(bronze, okta, hc, MIN_CREATION_DATE, use_old_okta_inner_join=True)
    print(
        "distinct okta_user_id — OLD inner:", distinct_count(silver_old, "okta_user_id"),
        "| NEW left+fallback:", distinct_count(silver_new, "okta_user_id"),
    )

In [ ]:
gold = build_gold_fact(silver_new)
print("Gold fact_copilot_active:")
display_df(gold.select("userId", "userEmail", "userFullEmail", "Request", "LastModification", "useCaseDisplayName"))
print("distinct userId:", distinct_count(gold, "userId"), "| distinct userEmail:", distinct_count(gold, "userEmail"))